# Test

## SIR 3S Toolkit

### Regular Import/Init

In [1]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [2]:
from sir3stoolkit.core import wrapper

In [3]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [4]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

### Additional Import/Init for Dataframes class

In [5]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [6]:
s3s = SIR3S_Model_Dataframes()

Initialization complete


## Additional

In [7]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx

# Open Model

In [8]:
s3s.OpenModel(dbName=r"../Tutorial051_Assets/Toolkit_Tutorial51_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

Model is open for further operation


# test

## Berechnung

In [9]:
tk_table_ALLG = s3s.GetTksofElementType(s3s.ObjectTypes_TableNames.ALLG)[0]

In [10]:
print(s3s.GetPropertiesofElementType(s3s.ObjectTypes_TableNames.ALLG))

['Name', 'Idqm', 'Idph', 'Netztyp', 'Forc', 'Pfadol1', 'Pk', 'InVariant', 'bz.Fk', 'bz.Cdat', 'bz.Cuhr', 'bz.Czon', 'bz.Iart', 'bz.ArtTh', 'bz.Thfakt', 'bz.Itrenn', 'bz.ThStat', 'bz.ThInst', 'bz.Jwarn', 'bz.Lfqsv', 'bz.Schwellqsig', 'bz.Knuvtyp', 'bz.ValidAggr', 'bz.CalcNetwork', 'bz.Idra', 'bz.CheckRes', 'bz.CheckMod']


### Rechenlauftyp

In [11]:
s3s.SetValue(Tk=tk_table_ALLG, propertyName="bz.Iart", Value="-1") # Daten prüfen
s3s.SetValue(Tk=tk_table_ALLG, propertyName="bz.Iart", Value="0") # stationär
s3s.SetValue(Tk=tk_table_ALLG, propertyName="bz.Iart", Value="1") # Lastfallschleife
s3s.SetValue(Tk=tk_table_ALLG, propertyName="bz.Iart", Value="2") # stationär+instationär

Value is set
Value is set
Value is set
Value is set


In [12]:
s3s.CloseModel(saveChangesBeforeClosing=True)

True

In [13]:
1/0

ZeroDivisionError: division by zero

In [ ]:
import sqlite3
from typing import Dict, List, Tuple

def diff_sqlite_db3(db1_path: str, db2_path: str) -> Dict[str, Dict[str, List[Tuple]]]:
    """
    Compare two SQLite databases and return differences.
    
    Returns a dictionary of structure:
    {
        "table_name": {
            "only_in_db1": [... rows ...],
            "only_in_db2": [... rows ...],
            "changed_rows": [... (db1_row, db2_row) ...]
        },
        ...
    }
    """

    conn1 = sqlite3.connect(db1_path)
    conn2 = sqlite3.connect(db2_path)

    cursor1 = conn1.cursor()
    cursor2 = conn2.cursor()

    # Get list of tables (assumes no internal sqlite tables)
    cursor1.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cursor1.fetchall()]

    diff_result = {}

    for table in tables:
        # Fetch all rows from both DBs
        cursor1.execute(f"SELECT * FROM {table}")
        cursor2.execute(f"SELECT * FROM {table}")

        rows1 = cursor1.fetchall()
        rows2 = cursor2.fetchall()

        # Convert lists to sets for diffing
        set1 = set(rows1)
        set2 = set(rows2)

        only_in_db1 = list(set1 - set2)
        only_in_db2 = list(set2 - set1)

        # Find rows that exist in both but differ in values.
        # Only possible if there is a primary key. We try to detect the first column as key.
        changed_rows = []
        if rows1 and rows2:
            # Get column names
            cursor1.execute(f"PRAGMA table_info({table})")
            columns_info = cursor1.fetchall()
            col_names = [c[1] for c in columns_info]

            # assume first column is primary key (common in most DBs)
            key_col = col_names[0]

            # build dicts keyed by first column
            map1 = {row[0]: row for row in rows1}
            map2 = {row[0]: row for row in rows2}

            for key in map1:
                if key in map2 and map1[key] != map2[key]:
                    changed_rows.append((map1[key], map2[key]))

        diff_result[table] = {
            "only_in_db1": only_in_db1,
            "only_in_db2": only_in_db2,
            "changed_rows": changed_rows,
        }

    conn1.close()
    conn2.close()

    return diff_result

In [ ]:
db_file_1 = r"C:\Users\aUsername\3S\SQLiteTabSync\Toolkit_Tutorial51_Model.db3"
db_file_2 = r"C:\Users\aUsername\3S\SQLiteTabSync\Toolkit_Tutorial51_Model_edited.db3"

In [ ]:
diffs = diff_sqlite_db3(db_file_1, db_file_2)

for table, result in diffs.items():
    print(f"\n=== Table: {table} ===")
    print("Only in DB1:", result["only_in_db1"])
    print("Only in DB2:", result["only_in_db2"])
    print("Changed rows:", result["changed_rows"])


=== Table: AB_DEF ===
Only in DB1: []
Only in DB2: []
Changed rows: []

=== Table: AGSN ===
Only in DB1: []
Only in DB2: []
Changed rows: []

=== Table: ALLG ===
Only in DB1: []
Only in DB2: []
Changed rows: []

=== Table: ALLG_BZ ===
Only in DB1: [('5762316468466095445', '5303199883422352888', '5597593673700177669', '13.02.2023', '00:00:00', 3, 11, 1, 1, 1, 0, 30, 1.0, 5.0, None, None, None, 0, 7, 7, '')]
Only in DB2: [('5762316468466095445', '5303199883422352888', '5597593673700177669', '13.02.2023', '00:00:00', 0, 11, 1, 1, 1, 0, 30, 1.0, 5.0, None, None, None, 0, 7, 7, '')]
Changed rows: [(('5762316468466095445', '5303199883422352888', '5597593673700177669', '13.02.2023', '00:00:00', 3, 11, 1, 1, 1, 0, 30, 1.0, 5.0, None, None, None, 0, 7, 7, ''), ('5762316468466095445', '5303199883422352888', '5597593673700177669', '13.02.2023', '00:00:00', 0, 11, 1, 1, 1, 0, 30, 1.0, 5.0, None, None, None, 0, 7, 7, ''))]

=== Table: ANTP ===
Only in DB1: []
Only in DB2: []
Changed rows: []

=== 

# ENUMS

In [ ]:
dir(s3s.ObjectTypes_TableNames)

['AGSN',
 'ALLG',
 'ANTP',
 'ANTP_ROWS',
 'ARRW',
 'ATMO',
 'AVOS',
 'AVOS_ROWS',
 'BEVE',
 'BEWI',
 'BREF',
 'CIRC',
 'CONT',
 'CRGL',
 'DPGR',
 'DPGR_DPKT',
 'DPKT',
 'DPRG',
 'DTRO',
 'DTRO_ROWD',
 'EBES',
 'ELEMENTQUERY',
 'ETAM',
 'ETAM_ROWS',
 'ETAR',
 'ETAR_ROWS',
 'ETAU',
 'ETAU_ROWS',
 'FKNL',
 'FQPS',
 'FQPS_BZ',
 'FSTF',
 'FWBZ',
 'FWEA',
 'FWES',
 'FWVB',
 'FWWU',
 'GKMP',
 'GMIX',
 'GRAV',
 'GTXT',
 'GVWK',
 'HAUS',
 'HYDR',
 'KLAP',
 'KNOT',
 'KOMK',
 'KOMK_ROWS',
 'KOMP',
 'LAYR',
 'LFKT',
 'LFKT_ROWT',
 'LTGR',
 'MREG',
 'NRCV',
 'NSCH',
 'OBEH',
 'OVAL',
 'PARI',
 'PARZ',
 'PGRP',
 'PGRP_PUMP',
 'PHI1',
 'PHI1_ROWT',
 'PHI2',
 'PHI2_ROWS',
 'PHIV',
 'PHIV_ROWS',
 'PHTR',
 'PLYG',
 'POLY',
 'PREG',
 'PUMD',
 'PUMD_ROWT',
 'PUMK',
 'PUMK_ROWS',
 'PUMP',
 'PVAR',
 'PVAR_ROWT',
 'PZON',
 'QVAR',
 'QVAR_ROWT',
 'RADD',
 'RART',
 'RCPL',
 'RCPL_ROWT',
 'RDIV',
 'RECT',
 'REGP',
 'REGV',
 'RFKT',
 'RHYS',
 'RINT',
 'RLSR',
 'RLVG',
 'RMES',
 'RMES_DPTS',
 'RMMA',
 'RMUL',
 'R

In [ ]:
s3s.GetPropertiesofElementType(s3s.ObjectTypes_TableNames.PARI)

['Name',
 'Nglopt',
 'Pk',
 'InVariant',
 'bz.Fk',
 'bz.Epsp',
 'bz.Epsqm',
 'bz.Epst',
 'bz.Epspreg',
 'bz.Epsqmreg',
 'bz.Epstrsp',
 'bz.Ntrspiter',
 'bz.Nziter',
 'bz.Ntiter']

In [ ]:
s3s.GetTksofElementType(s3s.ObjectTypes_TableNames.PARI)

['5711960102265934458']

In [ ]:
s3s.GetValue("5711960102265934458", "Name")

('PARI pk=5711960102265934458', 'string')

In [ ]:
s3s.GetTableRows()

TypeError: SIR3S_Model.GetTableRows() missing 1 required positional argument: 'tablePkTk'

In [ ]:
s3s.to_dotnet_enum(s3s.ObjectTypes_TableNames)

TypeError: Expected Enum member, got <class 'sir3stoolkit.core.wrapper.DotNetEnumMeta'>

In [ ]:
print(s3s.ProviderTypes)

<enum 'ProviderTypes'>


In [ ]:
1/0

ZeroDivisionError: division by zero

# Test

In [ ]:
import sys
import clr as net

In [ ]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [ ]:

sys.path.append(SIR3S_SIRGRAF_DIR)

net.AddReference(r"System")

net.AddReference(SIR3S_SIRGRAF_DIR+r"\Sir3S_Repository.Utilities")

# THE COMPILED DLL FOR Sir3S_Toolkit SHOULD ALSO BE COPIED TO SIR3S_SIRGRAF_DIR
net.AddReference(SIR3S_SIRGRAF_DIR+r"\Sir3S_Toolkit")

In [ ]:
import Sir3S_Repository.Utilities as Util
import Sir3S_Toolkit.Model as Sir3SToolkit